# Data Preparation (Posts + Reels + Analytics)

Prepare high-quality datasets from:
- `datasets/raw/Instagram_Posts.csv`
- `datasets/raw/Instagram_Reels.csv`
- `datasets/raw/Instagram_Analytics.csv`

for analysis of post characteristics and virality.


## Preparation Strategy

We follow a rigorous data science preparation pipeline:

1. **Load raw data** from source files.
2. **Profile** data quality (schema, missingness, duplicates, invalid values).
3. **Structure** fields into consistent formats (e.g., datetime, hashtag list).
4. **Clean** invalid values without introducing bias (avoid replacing unknowns with zeros).
5. **Enrich** features useful for later analysis (`has_hashtags`, `hashtag_count`).
6. **Validate** post-clean quality with explicit checks.
7. **Export** prepared datasets for reproducible work.

We prepare each dataset separately and will only create an optional "common-schema" combined table for compatible columns.


In [2]:
# Imports
import ast
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
# Ensure paths exist

# ROOT = /SC3021-SDAD-DS
ROOT = Path()

INSTAGRAM_POSTS_PATH = ROOT / "datasets" / "raw" / "Instagram_Posts.csv"
INSTAGRAM_REELS_PATH = ROOT / "datasets" / "raw" / "Instagram_Reels.csv"
OUTPUT_DIR = ROOT / "datasets" / "processed"

print("Posts path exists:", INSTAGRAM_POSTS_PATH.exists(), " | Path:", INSTAGRAM_POSTS_PATH)
print("Reels path exists:", INSTAGRAM_REELS_PATH.exists(), " | Path:", INSTAGRAM_REELS_PATH)
print("Output directory:", OUTPUT_DIR.resolve())

Posts path exists: True  | Path: datasets\raw\Instagram_Posts.csv
Reels path exists: True  | Path: datasets\raw\Instagram_Reels.csv
Output directory: D:\Z NTU\SC3021-SDAD-DS\datasets\processed


## 1) Load Raw Data

We first load both datasets exactly as provided, preserving raw values for profiling.


In [4]:
df_instagram_posts_raw = pd.read_csv(INSTAGRAM_POSTS_PATH)
df_instagram_reels_raw = pd.read_csv(INSTAGRAM_REELS_PATH)

print("Posts shape (Rows, Cols):", df_instagram_posts_raw.shape)
print("Reels shape (Rows, Cols):", df_instagram_reels_raw.shape)


Posts shape (Rows, Cols): (1000, 40)
Reels shape (Rows, Cols): (1000, 27)


## 2) Data Profiling Utilities

Items to Check:
- Data Types
- Nulls/Blanks
- Negatives/Zeros
- Uniqueness/Duplicates
- Value Distributions
- Outliers/Anomalies
- Pattern Analysis (Validate if data adheres to expected formats)
- Standardisation (Check for inconsistent representations of the same data, such as "USA", "US", "America")

In [ ]:
def check_data_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inspect the inferred data type of every column.

    Returns a DataFrame with one row per column showing:
    - dtype       : pandas inferred dtype (e.g. int64, float64, object)
    - unique_count: number of distinct values including NaN
    - sample_value: first non-null value for a quick sanity-check
    """
    rows = []
    for col in df.columns:
        series = df[col]
        non_null = series.dropna()
        rows.append({
            "column": col,
            "dtype": str(series.dtype),
            "unique_count": int(series.nunique(dropna=False)),
            "sample_value": non_null.iloc[0] if not non_null.empty else None,
        })
    return pd.DataFrame(rows).set_index("column")


def check_nulls_blanks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identify missing values (NaN/None) and blank strings in every column.

    Returns a DataFrame sorted by null_pct descending with columns:
    - null_count  : number of NaN / None values
    - null_pct    : percentage of rows that are null
    - blank_count : number of empty-string ("") values (object columns only)
    - blank_pct   : percentage of rows that are blank strings
    """
    n = len(df)
    rows = []
    for col in df.columns:
        series = df[col]
        null_count = int(series.isna().sum())
        blank_count = int((series == "").sum()) if series.dtype == object else 0
        rows.append({
            "column": col,
            "null_count": null_count,
            "null_pct": round(null_count / n * 100, 2),
            "blank_count": blank_count,
            "blank_pct": round(blank_count / n * 100, 2),
        })
    return (
        pd.DataFrame(rows)
        .set_index("column")
        .sort_values("null_pct", ascending=False)
    )


def check_negatives_zeros(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect negative and zero values across all numeric columns.

    Negative values flag impossible entries (e.g. negative likes) that
    should be treated as invalid rather than coerced to zero.
    Zero values flag potential silent fill-ins where NaN would be
    more honest.

    Returns a DataFrame sorted by neg_count descending with columns:
    - neg_count  : number of values < 0
    - neg_pct    : percentage of rows that are negative
    - zero_count : number of values == 0
    - zero_pct   : percentage of rows that are zero
    """
    n = len(df)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    rows = []
    for col in numeric_cols:
        series = df[col]
        neg_count = int((series < 0).sum())
        zero_count = int((series == 0).sum())
        rows.append({
            "column": col,
            "neg_count": neg_count,
            "neg_pct": round(neg_count / n * 100, 2),
            "zero_count": zero_count,
            "zero_pct": round(zero_count / n * 100, 2),
        })
    return (
        pd.DataFrame(rows)
        .set_index("column")
        .sort_values("neg_count", ascending=False)
    )


def check_uniqueness_duplicates(
    df: pd.DataFrame, id_cols: list[str] | None = None
) -> dict:
    """
    Assess row-level and column-level uniqueness.

    Checks:
    - fully duplicated rows (every column identical)
    - duplicate values within each supplied id_col (should be 0 for keys)
    - unique-value ratio per column (low ratio signals near-constant columns)

    Returns a dict with keys:
    - 'full_duplicate_rows' : int count of fully duplicated rows
    - 'id_duplicates'       : {col: duplicate_count} for each id_col found
    - 'uniqueness_ratio'    : DataFrame of unique_count and unique_pct per column
    """
    result = {}
    result["full_duplicate_rows"] = int(df.duplicated().sum())

    id_duplicates = {}
    for col in (id_cols or []):
        if col in df.columns:
            id_duplicates[col] = int(df[col].duplicated().sum())
    result["id_duplicates"] = id_duplicates

    n = len(df)
    rows = []
    for col in df.columns:
        u = int(df[col].nunique(dropna=False))
        rows.append({"column": col, "unique_count": u, "unique_pct": round(u / n * 100, 2)})
    result["uniqueness_ratio"] = pd.DataFrame(rows).set_index("column")

    return result


def check_value_distributions(df: pd.DataFrame, top_n: int = 10) -> dict:
    """
    Summarise how values are distributed across columns.

    For numeric columns : returns descriptive statistics
    (count, mean, std, min, quartiles, max) via describe().
    For categorical/object columns : returns the top_n most
    frequent values and their counts via value_counts().

    Returns a dict with keys:
    - 'numeric_summary'  : pd.DataFrame (transposed describe)
    - 'categorical_top'  : {col: pd.Series of top_n value counts}
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    numeric_summary = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()

    categorical_top = {}
    for col in cat_cols:
        categorical_top[col] = df[col].value_counts(dropna=False).head(top_n)

    return {"numeric_summary": numeric_summary, "categorical_top": categorical_top}


def check_outliers_anomalies(
    df: pd.DataFrame, method: str = "iqr", z_thresh: float = 3.0
) -> pd.DataFrame:
    """
    Detect statistical outliers in numeric columns.

    Two detection methods are supported:
    - 'iqr'    : flags values outside Q1 - 1.5*IQR or Q3 + 1.5*IQR (Tukey fences)
    - 'zscore' : flags values whose absolute z-score exceeds z_thresh

    Returns a DataFrame sorted by outlier_count descending with columns:
    - outlier_count : number of flagged values
    - outlier_pct   : percentage of rows flagged as outliers
    """
    n = len(df)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    rows = []
    for col in numeric_cols:
        series = df[col].dropna()
        if method == "iqr":
            q1, q3 = series.quantile(0.25), series.quantile(0.75)
            iqr = q3 - q1
            mask = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
        else:
            mean, std = series.mean(), series.std()
            mask = (
                ((df[col] - mean).abs() / std) > z_thresh
                if std > 0
                else pd.Series(False, index=df.index)
            )
        outlier_count = int(mask.sum())
        rows.append({
            "column": col,
            "outlier_count": outlier_count,
            "outlier_pct": round(outlier_count / n * 100, 2),
        })
    return (
        pd.DataFrame(rows)
        .set_index("column")
        .sort_values("outlier_count", ascending=False)
    )


def check_pattern_analysis(
    df: pd.DataFrame, patterns: dict[str, str] | None = None
) -> pd.DataFrame:
    """
    Validate that string column values conform to expected regex patterns.

    The patterns argument maps column names to regex strings, e.g.:
        {"url": r"^https?://", "post_id": r"^\\d+$"}
    Only non-null values are tested; nulls are excluded from the denominator.
    If patterns is None, a sensible default is applied for common column
    names (url, post_id).

    Returns a DataFrame with columns:
    - pattern        : the regex used
    - match_count    : values that matched
    - mismatch_count : values that did not match
    - mismatch_pct   : percentage of non-null values that failed
    """
    default_patterns: dict[str, str] = {
        "url": r"^https?://",
        "post_id": r"^\d+$",
    }
    patterns = patterns or default_patterns

    rows = []
    for col, pattern in patterns.items():
        if col not in df.columns:
            continue
        non_null = df[col].dropna().astype(str)
        match_mask = non_null.str.match(pattern)
        match_count = int(match_mask.sum())
        mismatch_count = int((~match_mask).sum())
        total = len(non_null)
        rows.append({
            "column": col,
            "pattern": pattern,
            "match_count": match_count,
            "mismatch_count": mismatch_count,
            "mismatch_pct": round(mismatch_count / total * 100, 2) if total > 0 else 0.0,
        })
    return pd.DataFrame(rows).set_index("column") if rows else pd.DataFrame()


def check_standardisation(
    df: pd.DataFrame, cols: list[str] | None = None
) -> dict:
    """
    Identify inconsistent representations of the same value in string columns.

    Detects values that differ only by:
    - leading/trailing whitespace (e.g. " USA" vs "USA")
    - letter casing (e.g. "usa", "USA", "Usa")

    For each checked column, groups raw values by their stripped-and-lowercased
    form. Groups with more than one raw variant indicate a standardisation
    opportunity.

    Returns a dict mapping column name to a DataFrame of variant groups:
    - normalised_key : the canonical (stripped + lowercased) form
    - raw_variants   : list of differing raw strings that map to it
    An empty DataFrame means no inconsistencies were found for that column.
    """
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    target_cols = [c for c in (cols or object_cols) if c in df.columns]

    result = {}
    for col in target_cols:
        series = df[col].dropna().astype(str)
        normalised = series.str.strip().str.lower()
        groups = (
            series.groupby(normalised)
            .apply(lambda g: sorted(g.unique().tolist()))
            .reset_index()
        )
        groups.columns = ["normalised_key", "raw_variants"]
        inconsistent = groups[groups["raw_variants"].apply(len) > 1].reset_index(drop=True)
        result[col] = inconsistent if not inconsistent.empty else pd.DataFrame()

    return result

In [ ]:
# instagram_posts_profile_raw = profile_dataframe(df_instagram_posts_raw, "Posts (Raw)")
# instagram_reels_profile_raw = profile_dataframe(df_instagram_reels_raw, "Reels (Raw)")

# display_profile(instagram_posts_profile_raw)
# display_profile(instagram_reels_profile_raw)

Dataset: Posts (Raw)
Shape: (1000, 40)
Duplicate post_id: 0
Duplicate url: 0
--------------------------------------------------------------------------------
Top missing columns:
                       missing_count  missing_pct
discovery_input                 1000        100.0
has_handshake                   1000        100.0
audio_url                        983         98.3
video_play_count                 894         89.4
coauthor_producers               893         89.3
engagement_score_view            878         87.8
product_type                     878         87.8
video_view_count                 834         83.4
videos                           821         82.1
videos_duration                  769         76.9
tagged_users                     730         73.0
location                         724         72.4
latest_comments                  417         41.7
hashtags                         353         35.3
alt_text                         132         13.2
---------------------

## 3) Structuring + Cleaning Utilities

Key principles:
- Convert numeric fields with `errors='coerce'`.
- Treat impossible negatives as invalid (`NaN`), not zero.
- Keep true missingness as missing (to avoid biasing medians/rates).
- Parse hashtag strings robustly into Python lists.


In [ ]:
def parse_hashtags(value):
    """Convert hashtag field into a clean list of hashtag strings."""
    if pd.isna(value):
        return []

    if isinstance(value, list):
        raw_list = value
    elif isinstance(value, str):
        text = value.strip()
        if text == "" or text == "[]":
            return []
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                raw_list = parsed
            else:
                # Fallback: split a plain comma string if parsing is not list-like
                raw_list = [x.strip() for x in text.split(",") if x.strip()]
        except (ValueError, SyntaxError):
            raw_list = [x.strip() for x in text.split(",") if x.strip()]
    else:
        return []

    cleaned = []
    for tag in raw_list:
        if tag is None:
            continue
        tag = str(tag).strip()
        if tag == "":
            continue
        cleaned.append(tag)
    return cleaned


def clean_nonnegative_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Coerce selected columns to numeric and set negative values to NaN."""
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
            out.loc[out[col] < 0, col] = np.nan
    return out


## 4) Prepare Posts Dataset (Separate Flow)


In [ ]:
posts = df_posts_raw.copy()
posts["source_type"] = "post"

# Datetime structuring
if "date_posted" in posts.columns:
    posts["date_posted"] = pd.to_datetime(posts["date_posted"], errors="coerce", utc=True)

# Deduplicate conservatively
if "post_id" in posts.columns:
    posts = posts.drop_duplicates(subset=["post_id"], keep="first")
elif "url" in posts.columns:
    posts = posts.drop_duplicates(subset=["url"], keep="first")

# Parse hashtags and engineer hashtag features
posts["hashtags_list"] = posts["hashtags"].apply(parse_hashtags) if "hashtags" in posts.columns else [[] for _ in range(len(posts))]
posts["has_hashtags"] = posts["hashtags_list"].apply(lambda x: len(x) > 0)
posts["hashtag_count"] = posts["hashtags_list"].apply(len)

# Clean core metrics (non-negative constraints)
posts = clean_nonnegative_numeric(posts, ["likes", "num_comments", "followers", "video_view_count", "video_play_count", "posts_count"])

# Track missingness flags for core metrics
for metric in ["likes", "num_comments"]:
    if metric in posts.columns:
        posts[f"{metric}_is_missing"] = posts[metric].isna()

print("Prepared posts shape:", posts.shape)
posts[["source_type", "post_id", "date_posted", "likes", "num_comments", "has_hashtags", "hashtag_count"]].head()


## 5) Prepare Reels Dataset (Separate Flow)


In [ ]:
reels = df_reels_raw.copy()
reels["source_type"] = "reel"

# Datetime structuring
if "date_posted" in reels.columns:
    reels["date_posted"] = pd.to_datetime(reels["date_posted"], errors="coerce", utc=True)

# Deduplicate conservatively
if "post_id" in reels.columns:
    reels = reels.drop_duplicates(subset=["post_id"], keep="first")
elif "url" in reels.columns:
    reels = reels.drop_duplicates(subset=["url"], keep="first")

# Parse hashtags and engineer hashtag features
reels["hashtags_list"] = reels["hashtags"].apply(parse_hashtags) if "hashtags" in reels.columns else [[] for _ in range(len(reels))]
reels["has_hashtags"] = reels["hashtags_list"].apply(lambda x: len(x) > 0)
reels["hashtag_count"] = reels["hashtags_list"].apply(len)

# Clean core metrics (non-negative constraints)
reels = clean_nonnegative_numeric(reels, ["likes", "num_comments", "views", "followers", "video_play_count", "following", "posts_count", "length"])

# Track missingness flags for core metrics
for metric in ["likes", "num_comments", "views", "followers"]:
    if metric in reels.columns:
        reels[f"{metric}_is_missing"] = reels[metric].isna()

print("Prepared reels shape:", reels.shape)
reels[["source_type", "post_id", "date_posted", "likes", "num_comments", "views", "followers", "has_hashtags", "hashtag_count"]].head()


## 6) Post-Clean Validation

We rerun quality checks after preparation to verify:
- duplicate handling worked
- impossible negative values were removed
- core engineered fields exist and look reasonable


In [ ]:
posts_profile_clean = profile_dataframe(posts, "Posts (Prepared)")
reels_profile_clean = profile_dataframe(reels, "Reels (Prepared)")

display_profile(posts_profile_clean)
display_profile(reels_profile_clean)


In [ ]:
print("Posts hashtag feature sanity check:")
print(posts["has_hashtags"].value_counts(dropna=False))
print(posts["hashtag_count"].describe())

print("\nReels hashtag feature sanity check:")
print(reels["has_hashtags"].value_counts(dropna=False))
print(reels["hashtag_count"].describe())


## 7) Optional Common-Schema Table (For Compatible Analyses Only)

A combined table can be useful for analyses based on shared columns (e.g., hashtags vs likes/comments),
but **should not** be used for metrics requiring Reel-only fields (`views`, `followers`) without careful handling.


In [ ]:
common_columns = [
    "source_type",
    "post_id",
    "url",
    "user_posted",
    "date_posted",
    "hashtags_list",
    "has_hashtags",
    "hashtag_count",
    "likes",
    "num_comments",
    "views",
    "followers",
    "is_verified",
    "is_paid_partnership",
]

def select_existing_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    out = df[existing].copy()
    for m in missing:
        out[m] = np.nan
    return out[cols]

posts_common = select_existing_columns(posts, common_columns)
reels_common = select_existing_columns(reels, common_columns)
df_common = pd.concat([posts_common, reels_common], ignore_index=True)

print("Common-schema combined shape:", df_common.shape)
df_common.head()


## 8) Export Prepared Outputs

We export:
- `posts_prepared.csv`
- `reels_prepared.csv`
- `combined_common_schema_prepared.csv` (optional shared schema)


In [ ]:
posts_out = OUTPUT_DIR / "posts_prepared.csv"
reels_out = OUTPUT_DIR / "reels_prepared.csv"
common_out = OUTPUT_DIR / "combined_common_schema_prepared.csv"

posts.to_csv(posts_out, index=False)
reels.to_csv(reels_out, index=False)
df_common.to_csv(common_out, index=False)

print("Saved:", posts_out)
print("Saved:", reels_out)
print("Saved:", common_out)


## 9) Data Preparation Summary (No Analysis Yet)

Completed:
- rigorous profiling for raw and cleaned data
- robust structuring/parsing of hashtags
- non-negative validation for engagement metrics
- explicit separation of Posts and Reels preparation pipelines
- reproducible exports for next-stage analysis/modeling

Not included in this notebook:
- statistical testing
- EDA conclusions
- predictive modeling

Those should be done in a separate analysis notebook using these prepared outputs.
